# Synova AI Fine-Tuning Pipeline
## Fine-tune Llama 3.1 8B on Synova-specific dataset
## Works on local machine, Colab, or any GPU environment

## Setup
Install required libraries

In [ ]:
# Install required libraries
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install"])

packages = [
    "torch",
    "transformers",
    "peft",
    "datasets",
    "bitsandbytes",
    "trl",
    "accelerate",
    "wandb"
]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

## Import Libraries

In [ ]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import json
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Configuration

In [ ]:
# Configuration
import os

# Model configuration
MODEL_NAME = os.getenv("MODEL_NAME", "meta-llama/Meta-Llama-3.1-8B")  # Base model
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./synova-finetuned")

# Training configuration
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "4"))  # Adjust based on GPU memory
GRADIENT_ACCUMULATION_STEPS = int(os.getenv("GRADIENT_ACCUMULATION_STEPS", "4"))
NUM_EPOCHS = int(os.getenv("NUM_EPOCHS", "3"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "2e-4"))
MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "512"))

# LoRA configuration (parameter-efficient fine-tuning)
LORA_R = int(os.getenv("LORA_R", "16"))
LORA_ALPHA = int(os.getenv("LORA_ALPHA", "32"))
LORA_DROPOUT = float(os.getenv("LORA_DROPOUT", "0.05"))
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

print("Configuration loaded:")
print(f"  Model: {MODEL_NAME}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")

# Load Dataset
import os
from pathlib import Path

# Define dataset path - use absolute path for Jupyter
DATASET_DIR = Path("c:/Users/McBuz/CascadeProjects/Synova AI Rebuild/synova-workspace/data/training")
DATASET_FILE = DATASET_DIR / "synova_dataset.jsonl"

# Try to load from local file
if DATASET_FILE.exists():
    def load_jsonl(file_path):
        data = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
        return data
    
    sample_data = load_jsonl(DATASET_FILE)
    print(f"Loaded {len(sample_data)} examples from {DATASET_FILE}")
else:
    # Create sample dataset for testing
    print(f"Dataset file not found at {DATASET_FILE}")
    print("Using sample dataset for testing...")
    sample_data = [
        {
            "instruction": "How should I implement JWT authentication in a Synova API endpoint?",
            "input": "",
            "output": "In Synova, JWT authentication is implemented using the middleware/auth.py module. Use the get_current_user dependency in your endpoint, and include the Authorization: Bearer <token> header in requests. The token is generated using the generate_token.py script."
        },
        {
            "instruction": "What is the recommended way to structure a new API router in Synova?",
            "input": "",
            "output": "Create a new file in apps/api/src/routers/ with an APIRouter instance. Define your endpoints with proper typing, error handling, and logging. Register the router in main.py with app.include_router(). Follow the patterns in existing routers like chat.py or agents.py."
        },
        {
            "instruction": "How do I configure environment variables for Synova?",
            "input": "",
            "output": "Environment variables are configured in .env.local for local development and .env.production for production. Key variables include OPENAI_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY, JWT_SECRET, and database credentials. Use python-dotenv to load these in your application."
        },
        {
            "instruction": "What is the PeakBrain class in Synova?",
            "input": "",
            "output": "PeakBrain is the central orchestration system in Synova that integrates multiple AI components including model routing, memory retrieval, governance checks, safety filtering, and tool execution. It processes BrainRequest objects and returns BrainResponse objects with full execution traces."
        }
    ]

print(f"\nTotal examples: {len(sample_data)}")
if sample_data:
    print(f"\nSample example:")
    print(json.dumps(sample_data[0], indent=2))

In [ ]:
# Load Dataset (replaced empty cell)

## Format Dataset for Training

In [ ]:
def format_example(example):
    """Format example for instruction tuning"""
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
    
    return prompt

# Format all examples
formatted_data = [format_example(ex) for ex in sample_data]

print(f"Formatted {len(formatted_data)} examples")
print(f"\nSample formatted example:")
print(formatted_data[0][:500] + "...")

## Load Model and Tokenizer

In [ ]:
# Load Model and Tokenizer
import os
from huggingface_hub import login

# Check for Hugging Face token
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_TOKEN")

if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("Logged into Hugging Face")
    except Exception as e:
        print(f"Warning: Could not login to Hugging Face: {e}")
else:
    print("Warning: No HF_TOKEN found. Using public model (may require acceptance of terms)")

# Load tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    print("Tokenizer loaded successfully")
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Please ensure you have accepted the model terms at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B")
    raise

# Try to load with quantization
try:
    from transformers import BitsAndBytesConfig
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN
    )
    print("Model loaded with 4-bit quantization")
except Exception as e:
    print(f"Error loading model with 4-bit quantization: {e}")
    print("Falling back to 8-bit quantization...")
    try:
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN
        )
        print("Model loaded with 8-bit quantization")
    except Exception as e2:
        print(f"Error loading model with 8-bit: {e2}")
        print("Falling back to full precision...")
        try:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=torch.float16,
                device_map="auto",
                trust_remote_code=True,
                token=HF_TOKEN
            )
            print("Model loaded with full precision")
        except Exception as e3:
            print(f"Error loading model: {e3}")
            raise

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)

print(f"Model loaded with LoRA")
print(f"Trainable parameters: {model.print_trainable_parameters()}")

## Tokenize Dataset

In [ ]:
# Tokenize Dataset
def tokenize_function(examples):
    """Tokenize a list of text examples"""
    return tokenizer(
        examples,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors=None
    )

# Tokenize formatted data
try:
    tokenized_data = tokenize_function(formatted_data)
    
    # Convert to dataset format
    from datasets import Dataset
    train_dataset = Dataset.from_dict(tokenized_data)
    
    # Split into train/validation
    if len(train_dataset) > 10:
        train_dataset = train_dataset.train_test_split(test_size=0.1)
        print(f"Train set size: {len(train_dataset['train'])}")
        print(f"Validation set size: {len(train_dataset['test'])}")
    else:
        print("Dataset too small for train/test split, using full dataset for training")
        train_dataset = {"train": train_dataset, "test": train_dataset}
        
except Exception as e:
    print(f"Error during tokenization: {e}")
    raise

## Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",  # Set to "wandb" if using Weights & Biases
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

print("Training arguments configured")

## Initialize Trainer

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal language modeling
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset["train"],
    eval_dataset=train_dataset["test"],
    data_collator=data_collator,
)

print("Trainer initialized")

## Start Training

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete!")

## Save Model

In [ ]:
# Save the fine-tuned model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")

## Save Model Locally

In [ ]:
# Save Model Locally
import shutil

# Create output directory if it doesn't exist
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Save the fine-tuned model
trainer.save_model(str(OUTPUT_PATH))
tokenizer.save_pretrained(str(OUTPUT_PATH))

print(f"Model saved to {OUTPUT_PATH}")

# Optional: Create a zip file for easy transfer
ZIP_PATH = OUTPUT_PATH.parent / f"{OUTPUT_PATH.name}.zip"
shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', str(OUTPUT_PATH))
print(f"Model zipped to {ZIP_PATH}")

## Optional: Upload to Hugging Face

In [ ]:
# Optional: Upload to Hugging Face
from huggingface_hub import login
import os

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_TOKEN")
HF_USERNAME = os.getenv("HF_USERNAME", "your-username")

if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        
        # Push to Hugging Face
        repo_id = f"{HF_USERNAME}/synova-llama-3.1-8b"
        print(f"Uploading model to {repo_id}...")
        
        model.push_to_hub(repo_id)
        tokenizer.push_to_hub(repo_id)
        
        print(f"Model successfully uploaded to https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"Error uploading to Hugging Face: {e}")
else:
    print("HF_TOKEN not set. Skipping Hugging Face upload.")
    print("To upload, set HF_TOKEN environment variable and re-run this cell.")